In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# load files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")

# Aggregation for metric.csv
metric_stats = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        count=('value', 'count'),
        mean_value=('value', 'mean'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
)

metric_ts = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(min_ts=('timestamp', 'min'), max_ts=('timestamp', 'max'))
)

metric_summary = (
    metric_stats
    .join(metric_ts)
    .reset_index()
)

# convert timestamps to timezone-aware datetimes in Asia/Shanghai (UTC+8)
metric_summary['min_ts'] = pd.to_datetime(metric_summary['min_ts'], unit='s', utc=True).dt.tz_convert(tz)
metric_summary['max_ts'] = pd.to_datetime(metric_summary['max_ts'], unit='s', utc=True).dt.tz_convert(tz)

# round numeric cols for compactness and sort by count desc, show top 20
metric_summary['mean_value'] = metric_summary['mean_value'].round(6)
metric_summary['p95_value'] = metric_summary['p95_value'].round(6)
metric_summary = metric_summary.sort_values('count', ascending=False).head(20)

# keep compact column order
metric_summary = metric_summary[['cmdb_id', 'kpi_name', 'count', 'mean_value', 'p95_value', 'min_ts', 'max_ts']]

# Aggregation for trace.csv
trace_stats = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        count=('value', 'count'),
        mean_value=('value', 'mean'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
)

trace_ts = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])
    .agg(min_ts=('timestamp', 'min'), max_ts=('timestamp', 'max'))
)

trace_summary = (
    trace_stats
    .join(trace_ts)
    .reset_index()
)

# convert timestamps to timezone-aware datetimes in Asia/Shanghai (UTC+8)
trace_summary['min_ts'] = pd.to_datetime(trace_summary['min_ts'], unit='s', utc=True).dt.tz_convert(tz)
trace_summary['max_ts'] = pd.to_datetime(trace_summary['max_ts'], unit='s', utc=True).dt.tz_convert(tz)

# round numeric cols for compactness and sort by count desc, show top 20
trace_summary['mean_value'] = trace_summary['mean_value'].round(6)
trace_summary['p95_value'] = trace_summary['p95_value'].round(6)
trace_summary = trace_summary.sort_values('count', ascending=False).head(20)

# keep compact column order
trace_summary = trace_summary[['cmdb_id', 'trace_name', 'count', 'mean_value', 'p95_value', 'min_ts', 'max_ts']]

# Display compact summaries (IPython will show these variables)
metric_summary, trace_summary

```
Out[1]:
```
```python
# Build a plain-English summary string from existing metric_summary and trace_summary variables
# (these were produced in the prior run)

# prepare top entries
top_metrics = metric_summary.head(5)
top_traces = trace_summary.head(5)

lines = []
lines.append("Summary of grouped telemetry (plain English):")
lines.append("")
lines.append(f"Metrics: showing top {len(top_metrics)} groups by sample count:")
for r in top_metrics.itertuples(index=False):
    cmdb, kpi, cnt, mean_v, p95_v, min_ts, max_ts = r
    lines.append(f"- {cmdb} / {kpi}: count={int(cnt)}, mean={mean_v}, p95={p95_v}, range={min_ts.isoformat()} to {max_ts.isoformat()}")

lines.append("")
lines.append(f"Traces: showing top {len(top_traces)} groups by sample count:")
for r in top_traces.itertuples(index=False):
    cmdb, trace_name, cnt, mean_v, p95_v, min_ts, max_ts = r
    lines.append(f"- {cmdb} / {trace_name}: count={int(cnt)}, mean={mean_v}, p95={p95_v}, range={min_ts.isoformat()} to {max_ts.isoformat()}")

lines.append("")
lines.append("Overall observations:")
# overall counts summary
metric_counts = metric_summary['count'].value_counts().sort_index(ascending=False)
trace_counts = trace_summary['count'].value_counts().sort_index(ascending=False)
lines.append(f"- Metric groups mainly have counts: {dict(metric_counts)} (most groups have 31 or 30 samples).")
lines.append(f"- Trace groups in the top results are primarily from docker_001 with counts: {dict(trace_counts)} (30 samples).")
lines.append("- Metric time window in top groups: mostly 2020-04-11 00:00 to 2020-04-11 00:30 (Asia/Shanghai, UTC+8).")
lines.append("- Trace time window in top groups: mostly 2020-04-11 00:00 to 2020-04-11 00:29 (Asia/Shanghai, UTC+8).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                  kpi_name  count  mean_value   p95_value                    min_ts                    max_ts
1629   os_020              Disk_io_util     31    0.000000    0.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
950    os_006                Send_total     31    0.000371    0.000472 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1050   os_008                Send_total     31    0.000579    0.000684 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1774  osb_001           app.succee_rate     31    0.993066    1.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1507   os_017                Send_total     31    0.195131    0.262561 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1751   os_022  Outgoing_network_traffic     31    0.050997    0.066615 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
5      db_001              CPU_free_pct     31   98.627717   98.955030 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
95     db_003              CPU_free_pct     31   97.321314   97.775516 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1773  osb_001            app.succee_num     31  372.370968  495.500000 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1771  osb_001              app.avg_time     31    0.002613    0.009355 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
971    os_007               Disk_rd_ios     31    0.000000    0.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1221   os_012              Disk_io_util     31    0.745413    2.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1386   os_015  Incoming_network_traffic     31    0.002193    0.002404 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
320    db_008              CPU_free_pct     31   97.172869   97.597032 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1247   os_012      Processor_load_5_min     31    0.009462    0.015000 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
288    db_007             MEM_real_util     31   78.589355   78.600000 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
1772  osb_001                   app.num     31  372.548387  495.500000 2020-04-11 00:00:00+08:00 2020-04-11 00:30:00+08:00
867    os_005             Disk_avgqu_sz     30    0.053414    0.032804 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
1589   os_019                 ICMP_ping     30    1.000000    1.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
858    os_004                  ss_total     30  369.100000  397.600000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00,        cmdb_id                                     trace_name  count   mean_value    p95_value                    min_ts                    max_ts
0   docker_001  trace.from_os_022.remoteprocess.duration_mean     30     0.527378     0.716308 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
1   docker_001   trace.from_os_022.remoteprocess.duration_p95     30     1.569500     2.175338 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
2   docker_001     trace.from_os_022.remoteprocess.error_rate     30     0.000000     0.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
3   docker_001      trace.from_os_022.remoteprocess.row_count     30    94.500000   124.950000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
4   docker_001                   trace.self.csf.duration_mean     30     0.049921     0.062178 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
5   docker_001                    trace.self.csf.duration_p95     30     0.146957     0.193000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
6   docker_001                      trace.self.csf.error_rate     30     0.000000     0.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
7   docker_001                       trace.self.csf.row_count     30   378.000000   499.800000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
8   docker_001             trace.self.flyremote.duration_mean     30     0.008512     0.012603 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
9   docker_001              trace.self.flyremote.duration_p95     30     0.028803     0.087425 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
10  docker_001                trace.self.flyremote.error_rate     30     0.000000     0.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
11  docker_001                 trace.self.flyremote.row_count     30    94.500000   124.950000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
12  docker_001                  trace.self.jdbc.duration_mean     30     0.002399     0.003171 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
13  docker_001                   trace.self.jdbc.duration_p95     30     0.003200     0.004000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
14  docker_001                     trace.self.jdbc.error_rate     30     0.000000     0.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
15  docker_001                      trace.self.jdbc.row_count     30  1513.933333  2002.400000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
16  docker_001                 trace.self.local.duration_mean     30     0.012076     0.017803 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
17  docker_001                  trace.self.local.duration_p95     30     0.074882     0.096550 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
18  docker_001                    trace.self.local.error_rate     30     0.000000     0.000000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00
19  docker_001                     trace.self.local.row_count     30  1323.000000  1749.300000 2020-04-11 00:00:00+08:00 2020-04-11 00:29:00+08:00)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing dataframes if present; otherwise load
try:
    df_metric
    df_trace
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_trace = pd.read_csv("trace.csv")

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using full metric series (rule 9)
metric_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'group_P95'})
)

# 2) Convert metric timestamps to timezone-aware datetimes in UTC+8 and filter the incident window (inclusive)
df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2020-04-11 00:00:00', tz=tz)
end = pd.Timestamp('2020-04-11 00:30:00', tz=tz)
df_metric_f = df_metric[(df_metric['ts_dt'] >= start) & (df_metric['ts_dt'] <= end)].copy()

# 3) Attach group_P95 to filtered metric rows
df_metric_f = df_metric_f.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Detect anomalies: value >= group_P95
df_metric_f['is_anom'] = df_metric_f['value'] >= df_metric_f['group_P95']

# 5) Aggregate anomalies per group
metric_anoms = df_metric_f[df_metric_f['is_anom']].copy()
if not metric_anoms.empty:
    metric_agg = (
        metric_anoms
        .groupby(['cmdb_id', 'kpi_name'])
        .agg(
            anomaly_count=('value', 'count'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            max_anomaly_value=('value', 'max')
        )
        .reset_index()
    )
    # first_anomaly_value: value at the earliest anomaly per group
    first_vals = (
        metric_anoms
        .sort_values(['timestamp', 'value'])
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .first()[['cmdb_id', 'kpi_name', 'value', 'timestamp']]
        .rename(columns={'value': 'first_anomaly_value', 'timestamp': 'earliest_ts_for_first_value'})
    )
    metric_agg = metric_agg.merge(first_vals[['cmdb_id','kpi_name','first_anomaly_value']], on=['cmdb_id','kpi_name'], how='left')
    # attach group_P95
    metric_agg = metric_agg.merge(metric_p95, on=['cmdb_id','kpi_name'], how='left')
    # convert earliest timestamp to tz-aware dt
    metric_agg['earliest_anomaly_timestamp'] = pd.to_datetime(metric_agg['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # round numeric fields for compactness
    metric_agg['group_P95'] = metric_agg['group_P95'].round(6)
    metric_agg['first_anomaly_value'] = metric_agg['first_anomaly_value'].round(6)
    metric_agg['max_anomaly_value'] = metric_agg['max_anomaly_value'].round(6)
    # select and sort
    metric_anom_summary = (
        metric_agg[['cmdb_id','kpi_name','group_P95','anomaly_count','earliest_anomaly_timestamp','first_anomaly_value','max_anomaly_value']]
        .sort_values('anomaly_count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    # empty summary with correct columns
    metric_anom_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','group_P95','anomaly_count','earliest_anomaly_timestamp','first_anomaly_value','max_anomaly_value'])

# -----------------------
# TRACE processing
# 1) global P95 per (cmdb_id, trace_name)
trace_p95 = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'group_P95'})
)

# 2) timestamps -> tz-aware and filter incident window
df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace_f = df_trace[(df_trace['ts_dt'] >= start) & (df_trace['ts_dt'] <= end)].copy()

# 3) attach group_P95
df_trace_f = df_trace_f.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')

# 4) anomaly definition:
# if trace_name contains 'error' or 'err' (case-insensitive) -> anomaly if value > 0
# OR value >= group_P95
is_err_name = df_trace_f['trace_name'].str.contains(r'error|err', case=False, na=False)
df_trace_f['is_err_name'] = is_err_name
df_trace_f['is_anom'] = (df_trace_f['is_err_name'] & (df_trace_f['value'] > 0)) | (df_trace_f['value'] >= df_trace_f['group_P95'])

# 5) aggregate anomalies per group
trace_anoms = df_trace_f[df_trace_f['is_anom']].copy()
if not trace_anoms.empty:
    trace_agg = (
        trace_anoms
        .groupby(['cmdb_id','trace_name'])
        .agg(
            anomaly_count=('value','count'),
            earliest_anomaly_timestamp=('timestamp','min'),
            max_anomaly_value=('value','max')
        )
        .reset_index()
    )
    first_vals_t = (
        trace_anoms
        .sort_values(['timestamp','value'])
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .first()[['cmdb_id','trace_name','value','timestamp']]
        .rename(columns={'value':'first_anomaly_value','timestamp':'earliest_ts_for_first_value'})
    )
    trace_agg = trace_agg.merge(first_vals_t[['cmdb_id','trace_name','first_anomaly_value']], on=['cmdb_id','trace_name'], how='left')
    trace_agg = trace_agg.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')
    trace_agg['earliest_anomaly_timestamp'] = pd.to_datetime(trace_agg['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    trace_agg['group_P95'] = trace_agg['group_P95'].round(6)
    trace_agg['first_anomaly_value'] = trace_agg['first_anomaly_value'].round(6)
    trace_agg['max_anomaly_value'] = trace_agg['max_anomaly_value'].round(6)
    trace_anom_summary = (
        trace_agg[['cmdb_id','trace_name','group_P95','anomaly_count','earliest_anomaly_timestamp','first_anomaly_value','max_anomaly_value']]
        .sort_values('anomaly_count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    trace_anom_summary = pd.DataFrame(columns=['cmdb_id','trace_name','group_P95','anomaly_count','earliest_anomaly_timestamp','first_anomaly_value','max_anomaly_value'])

# Display compact summaries (IPython display via variable names)
metric_anom_summary, trace_anom_summary

```
Out[2]:
```
```python
# Build a plain-English summary string from existing metric_anom_summary and trace_anom_summary variables
# (these were produced in the prior run). Display a compact human-readable summary.

try:
    metric_anom_summary
    trace_anom_summary
except NameError:
    import pandas as pd
    metric_anom_summary = pd.DataFrame()
    trace_anom_summary = pd.DataFrame()

lines = []
lines.append("Anomaly summary (incident window 2020-04-11 00:00:00+08 to 2020-04-11 00:30:00+08):")
lines.append("")

# Metrics summary
n_metric = len(metric_anom_summary)
lines.append(f"Metrics: {n_metric} group(s) detected with anomalies (showing top 5):")
for r in metric_anom_summary.head(5).itertuples(index=False):
    cmdb, kpi, p95, cnt, earliest_ts, first_val, max_val = r
    # format numbers compactly
    p95_s = f"{p95:.6g}" if pd.notna(p95) else "NA"
    first_s = f"{first_val:.6g}" if pd.notna(first_val) else "NA"
    max_s = f"{max_val:.6g}" if pd.notna(max_val) else "NA"
    lines.append(f"- {cmdb} / {kpi}: anomalies={int(cnt)}, group_P95={p95_s}, earliest={earliest_ts.isoformat()}, first={first_s}, max={max_s}")

if n_metric == 0:
    lines.append("- (no metric anomalies detected)")

lines.append("")

# Traces summary
n_trace = len(trace_anom_summary)
lines.append(f"Traces: {n_trace} group(s) detected with anomalies (showing top 5):")
for r in trace_anom_summary.head(5).itertuples(index=False):
    cmdb, trace_name, p95, cnt, earliest_ts, first_val, max_val = r
    p95_s = f"{p95:.6g}" if pd.notna(p95) else "NA"
    first_s = f"{first_val:.6g}" if pd.notna(first_val) else "NA"
    max_s = f"{max_val:.6g}" if pd.notna(max_val) else "NA"
    lines.append(f"- {cmdb} / {trace_name}: anomalies={int(cnt)}, group_P95={p95_s}, earliest={earliest_ts.isoformat()}, first={first_s}, max={max_s}")

if n_trace == 0:
    lines.append("- (no trace anomalies detected)")

lines.append("")
# Key observations
lines.append("Key observations:")
lines.append("- Many top metric groups (e.g., Disk_io_util, Disk_rd_ios, Redis and DB counters) show anomaly_count equal to their full sample counts in the window (30 or 31).")
lines.append("- Several groups have group_P95 = 0; in those cases values equal to 0 meet the '>= P95' rule, so zero-valued points were flagged as anomalies.")
lines.append("- Trace anomalies are dominated by error_rate series from docker_* (many have group_P95 = 0 and anomaly_count = 30). Note those series' first and max anomaly values are 0, indicating no non-zero error values but flagged due to group_P95 being zero.")
lines.append("- All anomalies occurred at or after 2020-04-11 00:00:00+08, within the requested incident window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id           kpi_name     group_P95  anomaly_count earliest_anomaly_timestamp  first_anomaly_value  max_anomaly_value
0      os_020       Disk_io_util  0.000000e+00             31  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
1      os_007        Disk_rd_ios  0.000000e+00             31  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
2      os_017          ICMP_ping  1.000000e+00             30  2020-04-11 00:00:00+08:00         1.000000e+00       1.000000e+00
3      os_016          ICMP_ping  1.000000e+00             30  2020-04-11 00:00:00+08:00         1.000000e+00       1.000000e+00
4      db_013               AIOS  0.000000e+00             30  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
5   redis_008       evicted_keys  0.000000e+00             30  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
6   redis_008  connected_clients  2.300000e+01             30  2020-04-11 00:00:00+08:00         2.300000e+01       2.400000e+01
7   redis_008    blocked_clients  0.000000e+00             30  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
8   redis_008    Redis_key_count  3.281000e+03             30  2020-04-11 00:00:00+08:00         3.281000e+03       3.281000e+03
9      db_007           Row_Lock  0.000000e+00             30  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
10     db_007               Hang  1.000000e+00             30  2020-04-11 00:00:00+08:00         1.000000e+00       1.000000e+00
11     db_007       On_Off_State  1.000000e+00             30  2020-04-11 00:00:00+08:00         1.000000e+00       1.000000e+00
12  redis_002         redis_ping  1.000000e+00             30  2020-04-11 00:00:00+08:00         1.000000e+00       1.000000e+00
13     db_009           Row_Lock  0.000000e+00             30  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
14  redis_002          maxmemory  2.000000e+09             30  2020-04-11 00:00:00+08:00         2.000000e+09       2.000000e+09
15  redis_002       expired_keys  2.000000e+00             30  2020-04-11 00:00:00+08:00         2.000000e+00       2.000000e+00
16  redis_002       evicted_keys  0.000000e+00             30  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
17  redis_007         redis_ping  1.000000e+00             30  2020-04-11 00:00:00+08:00         1.000000e+00       1.000000e+00
18  redis_007         redis_load  0.000000e+00             30  2020-04-11 00:00:00+08:00         0.000000e+00       0.000000e+00
19  redis_008       expired_keys  5.100000e+02             30  2020-04-11 00:00:00+08:00         5.100000e+02       5.100000e+02,        cmdb_id                                    trace_name  group_P95  anomaly_count earliest_anomaly_timestamp  first_anomaly_value  max_anomaly_value
0   docker_001    trace.from_os_022.remoteprocess.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
1   docker_001                     trace.self.csf.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
2   docker_001                    trace.self.jdbc.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
3   docker_001               trace.self.flyremote.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
4   docker_003                   trace.self.local.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
5   docker_003  trace.to_docker_005.remoteprocess.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
6   docker_003                    trace.self.jdbc.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
7   docker_003               trace.self.flyremote.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
8   docker_002                     trace.self.csf.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
9   docker_002               trace.self.flyremote.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
10  docker_002                   trace.self.local.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
11  docker_002                    trace.self.jdbc.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
12  docker_001  trace.to_docker_008.remoteprocess.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
13  docker_002    trace.from_os_022.remoteprocess.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
14  docker_001  trace.to_docker_007.remoteprocess.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
15  docker_001                   trace.self.local.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
16  docker_003    trace.from_os_021.remoteprocess.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
17  docker_003                     trace.self.csf.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
18  docker_002  trace.to_docker_008.remoteprocess.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0
19  docker_002  trace.to_docker_007.remoteprocess.error_rate        0.0             30  2020-04-11 00:00:00+08:00                  0.0                0.0)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from collections import namedtuple

tz = pytz.timezone('Asia/Shanghai')

# reuse existing df_metric if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp columns exist and types
if 'timestamp' not in df_metric.columns:
    raise ValueError("metric.csv must contain 'timestamp' column")

# Compute global P95 per (cmdb_id, kpi_name) using full series (rule 9)
metric_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'group_P95'})
)

# Also compute global info to detect binary KPIs (using full series for db_007)
db_full = df_metric[df_metric['cmdb_id'] == 'db_007'].copy()

# Incident window bounds in UTC+8
start = pd.Timestamp('2020-04-11 00:00:00', tz=tz)
end = pd.Timestamp('2020-04-11 00:30:00', tz=tz)

# Prepare filtered db_007 rows in the incident window (convert timestamps to tz-aware datetimes)
db_full['ts_dt'] = pd.to_datetime(db_full['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
db_window = db_full[(db_full['ts_dt'] >= start) & (db_full['ts_dt'] <= end)].copy()
# Keep integer timestamp for consecutive-checks
db_window['ts_int'] = db_window['timestamp'].astype(int)

# If no db_007 data, return empty
if db_full.empty:
    db007_summary = pd.DataFrame(columns=['kpi_name','group_P95','total_anomaly_count',
                                         'earliest_anomaly_timestamp','longest_consecutive_anomaly_length',
                                         'start_timestamp_of_longest_consecutive_anomaly','max_value_within_longest_consecutive_anomaly'])
else:
    results = []

    # get list of KPIs for db_007 from global full series
    kpis = db_full['kpi_name'].unique()
    for kpi in kpis:
        # global group_P95 (from metric_p95)
        row = metric_p95[(metric_p95['cmdb_id']=='db_007') & (metric_p95['kpi_name']==kpi)]
        group_P95 = float(row['group_P95'].iloc[0]) if not row.empty else float('nan')

        # detect binary KPI using full db_007 series for this kpi
        full_vals = db_full[db_full['kpi_name']==kpi]['value']
        unique_vals = pd.unique(full_vals.dropna())
        is_binary = (len(unique_vals) <= 2) and set(unique_vals).issubset({0,1})

        # rows in incident window for this kpi
        grp = db_window[db_window['kpi_name']==kpi].sort_values('ts_int').copy()

        # define anomaly rule per instructions:
        # - if binary KPI: anomaly if value == 1
        # - else if group_P95 == 0: anomaly if value > 0
        # - else anomaly if value >= group_P95
        if is_binary:
            grp['is_anom'] = grp['value'] == 1
        else:
            if pd.isna(group_P95):
                grp['is_anom'] = False
            else:
                if group_P95 == 0:
                    grp['is_anom'] = grp['value'] > 0
                else:
                    grp['is_anom'] = grp['value'] >= group_P95

        total_anomaly_count = int(grp['is_anom'].sum())

        if total_anomaly_count == 0:
            earliest_anom_ts = pd.NaT
            longest_run_len = 0
            start_ts_longest = pd.NaT
            max_val_in_longest = float('nan')
        else:
            # earliest anomaly timestamp (tz-aware)
            earliest_row = grp[grp['is_anom']].nsmallest(1, 'ts_int').iloc[0]
            earliest_anom_ts = pd.to_datetime(int(earliest_row['timestamp']), unit='s', utc=True).tz_convert(tz)

            # Determine longest consecutive anomaly run where consecutive timestamps differ by 60 seconds
            # Iterate over anomaly rows in order
            anom_rows = grp[grp['is_anom']][['timestamp','ts_int','value']].sort_values('ts_int').to_dict('records')
            longest_run_len = 0
            current_len = 0
            current_start_ts = None
            current_max_val = None
            best_start_ts = None
            best_max_val = None
            prev_ts = None
            for rec in anom_rows:
                ts = int(rec['ts_int'])
                val = float(rec['value'])
                if prev_ts is None:
                    # start first run
                    current_len = 1
                    current_start_ts = ts
                    current_max_val = val
                else:
                    if ts == prev_ts + 60:
                        current_len += 1
                        if val > current_max_val:
                            current_max_val = val
                    else:
                        # finalize previous run
                        if current_len > longest_run_len:
                            longest_run_len = current_len
                            best_start_ts = current_start_ts
                            best_max_val = current_max_val
                        # start new run
                        current_len = 1
                        current_start_ts = ts
                        current_max_val = val
                prev_ts = ts
            # finalize last run
            if current_len > longest_run_len:
                longest_run_len = current_len
                best_start_ts = current_start_ts
                best_max_val = current_max_val

            # convert best_start_ts to tz-aware
            start_ts_longest = pd.to_datetime(int(best_start_ts), unit='s', utc=True).tz_convert(tz)
            max_val_in_longest = float(best_max_val)

        results.append({
            'kpi_name': kpi,
            'group_P95': group_P95,
            'total_anomaly_count': total_anomaly_count,
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'longest_consecutive_anomaly_length': int(longest_run_len),
            'start_timestamp_of_longest_consecutive_anomaly': start_ts_longest,
            'max_value_within_longest_consecutive_anomaly': max_val_in_longest
        })

    db007_summary = pd.DataFrame(results)

    # formatting: round numeric columns compactly
    db007_summary['group_P95'] = db007_summary['group_P95'].round(6)
    db007_summary['max_value_within_longest_consecutive_anomaly'] = db007_summary['max_value_within_longest_consecutive_anomaly'].round(6)

    # sort by longest_consecutive_anomaly_length desc and limit to top 20
    db007_summary = db007_summary.sort_values('longest_consecutive_anomaly_length', ascending=False).head(20).reset_index(drop=True)

# Display compact summary for db_007 KPIs
db007_summary

```
Out[3]:
```
```python
# Build a plain-English summary string from the existing db007_summary variable and display it.
# (db007_summary was produced in the prior run)

try:
    db007_summary
except NameError:
    db007_summary = None

if db007_summary is None or db007_summary.empty:
    summary = "No db_007 KPI data available in the workspace to summarize."
else:
    lines = []
    lines.append("db_007 KPI anomaly summary (window 2020-04-11 00:00:00+08 to 2020-04-11 00:30:00+08):")
    lines.append("")
    # Top findings by longest consecutive anomaly length (already sorted)
    top = db007_summary.head(10)
    for r in top.itertuples(index=False):
        kpi = r.kpi_name
        p95 = r.group_P95
        total = int(r.total_anomaly_count)
        earliest = r.earliest_anomaly_timestamp
        run_len = int(r.longest_consecutive_anomaly_length)
        run_start = r.start_timestamp_of_longest_consecutive_anomaly
        run_max = r.max_value_within_longest_consecutive_anomaly
        lines.append(f"- {kpi}: group_P95={p95:.6g}, anomalies_in_window={total}, earliest={earliest},")
        lines.append(f"  longest_consecutive_run={run_len} min starting {run_start}, max_within_run={run_max:.6g}")
    lines.append("")
    # Key observations
    lines.append("Key observations:")
    lines.append("- Hang and On_Off_State are binary flags and were anomalous for the entire window (30 min each, value==1).")
    lines.append("- MEM_Total also flagged anomalous for the entire window (30 min). Investigate memory sizing/measurement.")
    lines.append("- Proc_User_Used_Pct and Proc_Used_Pct show a sustained anomaly run of 11 minutes starting ~2020-04-11 00:05+08 (13 total anomaly points).")
    lines.append("- Sess_Connect had a 9-minute continuous anomaly run starting ~00:07+08 (9 anomalies total).")
    lines.append("- Other KPIs (AWS, MEM_real_util, tnsping_result_time, etc.) show shorter runs or isolated anomalies.")
    lines.append("")
    lines.append("Immediate next checks:")
    lines.append("- Prioritize investigating Hang and On_Off_State (continuous binary flags) and MEM_Total (continuous high) for db_007.")
    lines.append("- Review process CPU/memory (Proc_*), session/connect behavior (Sess_Connect), and tnsping/DB response metrics around 00:05–00:10+08.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name    group_P95  total_anomaly_count earliest_anomaly_timestamp  longest_consecutive_anomaly_length start_timestamp_of_longest_consecutive_anomaly  max_value_within_longest_consecutive_anomaly
0                  Hang     1.000000                   30  2020-04-11 00:00:00+08:00                                  30                      2020-04-11 00:00:00+08:00                                      1.000000
1          On_Off_State     1.000000                   30  2020-04-11 00:00:00+08:00                                  30                      2020-04-11 00:00:00+08:00                                      1.000000
2             MEM_Total   506.929929                   30  2020-04-11 00:00:00+08:00                                  30                      2020-04-11 00:00:00+08:00                                    506.929929
3    Proc_User_Used_Pct     2.860000                   13  2020-04-11 00:05:00+08:00                                  11                      2020-04-11 00:05:00+08:00                                      2.870000
4         Proc_Used_Pct     2.860000                   13  2020-04-11 00:05:00+08:00                                  11                      2020-04-11 00:05:00+08:00                                      2.870000
5          Sess_Connect   394.000000                    9  2020-04-11 00:07:00+08:00                                   9                      2020-04-11 00:07:00+08:00                                    394.000000
6                   AWS     0.020000                   15  2020-04-11 00:04:00+08:00                                   4                      2020-04-11 00:09:00+08:00                                      0.020000
7         MEM_real_util    78.600000                    8  2020-04-11 00:01:00+08:00                                   4                      2020-04-11 00:27:00+08:00                                     78.600000
8   tnsping_result_time    10.000000                    7  2020-04-11 00:00:00+08:00                                   3                      2020-04-11 00:09:00+08:00                                     10.000000
9          CPU_free_pct    96.708572                    2  2020-04-11 00:01:00+08:00                                   2                      2020-04-11 00:01:00+08:00                                     97.225609
10         CPU_Used_Pct     4.263050                    2  2020-04-11 00:06:00+08:00                                   1                      2020-04-11 00:06:00+08:00                                      4.265833
11                 AIOS     0.010000                    3  2020-04-11 00:08:00+08:00                                   1                      2020-04-11 00:08:00+08:00                                      0.010000
12                  ACS     0.025500                    2  2020-04-11 00:22:00+08:00                                   1                      2020-04-11 00:22:00+08:00                                      0.030000
13   Logic_Read_Per_Sec  2670.185000                    2  2020-04-11 00:06:00+08:00                                   1                      2020-04-11 00:06:00+08:00                                  12008.900000
14       LFSync_Per_Sec     1.656500                    2  2020-04-11 00:15:00+08:00                                   1                      2020-04-11 00:15:00+08:00                                      1.740000
15  LFParaWrite_Per_Sec     1.404000                    2  2020-04-11 00:06:00+08:00                                   1                      2020-04-11 00:06:00+08:00                                      1.750000
16         Exec_Per_Sec   122.220000                    2  2020-04-11 00:18:00+08:00                                   1                      2020-04-11 00:18:00+08:00                                    126.000000
17  DFParaWrite_Per_Sec     1.758500                    2  2020-04-11 00:08:00+08:00                                   1                      2020-04-11 00:08:00+08:00                                      1.790000
18               DbTime    22.390000                    2  2020-04-11 00:18:00+08:00                                   1                      2020-04-11 00:18:00+08:00                                     23.200000
19         Call_Per_Sec    63.240000                    2  2020-04-11 00:12:00+08:00                                   1                      2020-04-11 00:12:00+08:00                                     89.900000```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse existing df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# 1) Identify trace groups containing 'jdbc' (case-insensitive) using full series
mask_jdbc = df_trace['trace_name'].str.contains('jdbc', case=False, na=False)
jdbc_groups = df_trace[mask_jdbc][['cmdb_id', 'trace_name']].drop_duplicates()

# 2) Compute global P95 per (cmdb_id, trace_name) using the full trace series (rule 9)
trace_p95_all = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'group_P95'})
)

# keep only jdbc-related groups' P95
trace_p95_jdbc = trace_p95_all.merge(jdbc_groups, on=['cmdb_id','trace_name'], how='inner')

# 3) Convert timestamps to timezone-aware datetimes in UTC+8 and filter to incident window (inclusive)
df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2020-04-11 00:00:00', tz=tz)
end = pd.Timestamp('2020-04-11 00:30:00', tz=tz)
df_trace_window = df_trace[(df_trace['ts_dt'] >= start) & (df_trace['ts_dt'] <= end)].copy()

# 4) Keep only jdbc-related rows in the window
df_trace_jdbc_win = df_trace_window[df_trace_window['trace_name'].str.contains('jdbc', case=False, na=False)].copy()

# 5) Attach group_P95 (computed from full series) to the filtered rows
df_trace_jdbc_win = df_trace_jdbc_win.merge(trace_p95_jdbc, on=['cmdb_id','trace_name'], how='left')

# 6) Mark rows as anomalous if value >= group_P95 OR (trace_name contains 'error' and value>0)
is_err_name = df_trace_jdbc_win['trace_name'].str.contains('error', case=False, na=False)
df_trace_jdbc_win['is_anom'] = (df_trace_jdbc_win['value'] >= df_trace_jdbc_win['group_P95']) | (is_err_name & (df_trace_jdbc_win['value'] > 0))

# 7) Aggregate per group for anomalies
anoms = df_trace_jdbc_win[df_trace_jdbc_win['is_anom']].copy()
if anoms.empty:
    trace_jdbc_anom_summary = pd.DataFrame(columns=['cmdb_id','trace_name','group_P95','anomaly_count',
                                                   'earliest_anomaly_timestamp','first_anomaly_value','max_anomaly_value'])
else:
    # earliest anomaly timestamp and first anomaly value per group
    earliest = (
        anoms
        .sort_values(['timestamp','value'])
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .first()[['cmdb_id','trace_name','timestamp','value']]
        .rename(columns={'timestamp':'earliest_anomaly_timestamp','value':'first_anomaly_value'})
    )
    # other aggregates
    agg = (
        anoms
        .groupby(['cmdb_id','trace_name'])
        .agg(
            anomaly_count=('value','count'),
            max_anomaly_value=('value','max')
        )
        .reset_index()
    )
    trace_jdbc_anom_summary = agg.merge(earliest, on=['cmdb_id','trace_name'], how='left')
    # attach group_P95
    trace_jdbc_anom_summary = trace_jdbc_anom_summary.merge(trace_p95_jdbc, on=['cmdb_id','trace_name'], how='left')
    # convert earliest timestamp to tz-aware datetime
    trace_jdbc_anom_summary['earliest_anomaly_timestamp'] = pd.to_datetime(trace_jdbc_anom_summary['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # round numeric columns compactly
    trace_jdbc_anom_summary['group_P95'] = trace_jdbc_anom_summary['group_P95'].round(6)
    trace_jdbc_anom_summary['first_anomaly_value'] = trace_jdbc_anom_summary['first_anomaly_value'].round(6)
    trace_jdbc_anom_summary['max_anomaly_value'] = trace_jdbc_anom_summary['max_anomaly_value'].round(6)
    # select requested columns and sort by earliest_anomaly_timestamp ascending, limit top 20
    trace_jdbc_anom_summary = trace_jdbc_anom_summary[['cmdb_id','trace_name','group_P95','anomaly_count',
                                                       'earliest_anomaly_timestamp','first_anomaly_value','max_anomaly_value']] \
                                    .sort_values('earliest_anomaly_timestamp', ascending=True).head(20).reset_index(drop=True)

# Display compact result variable
trace_jdbc_anom_summary

```
Out[4]:
```
```python
# Build a concise plain-English summary from trace_jdbc_anom_summary produced earlier.
try:
    trace_jdbc_anom_summary
except NameError:
    import pandas as pd
    trace_jdbc_anom_summary = pd.DataFrame()

if trace_jdbc_anom_summary.empty:
    summary = "No JDBC trace anomalies detected in the provided data/window."
else:
    lines = []
    lines.append("JDBC trace anomaly summary (window 2020-04-11 00:00–00:30+08):")
    lines.append("")
    # top rows are already sorted by earliest_anomaly_timestamp ascending
    lines.append(f"- {len(trace_jdbc_anom_summary)} JDBC trace groups had >=1 anomaly in the window. Showing key points:")
    lines.append("")
    # highlight groups with purely-zero error_rate flags vs meaningful latency anomalies
    zero_error_rows = trace_jdbc_anom_summary[
        trace_jdbc_anom_summary['trace_name'].str.contains('error_rate', case=False, na=False) &
        (trace_jdbc_anom_summary['group_P95'] == 0)
    ]
    if not zero_error_rows.empty:
        lines.append(f"- Many groups (e.g., {len(zero_error_rows)} entries) are 'error_rate' series with group_P95=0 and anomaly_count=30; their first/max anomaly values are 0.0. These are likely artifacts of the >=P95 rule (zeros flagged) rather than real non-zero errors.")
    # call out notable latency (duration) anomalies with non-zero max values
    duration_anoms = trace_jdbc_anom_summary[trace_jdbc_anom_summary['trace_name'].str.contains('duration', case=False, na=False)]
    if not duration_anoms.empty:
        lines.append("- Duration-related JDBC anomalies with non-zero values (higher priority to investigate):")
        for r in duration_anoms.itertuples(index=False):
            cmdb, tname, p95, cnt, earliest, first_val, max_val = r
            lines.append(f"  • {cmdb} / {tname}: group_P95={p95:.6g}, anomalies={int(cnt)}, earliest={earliest}, first={first_val:.6g}, max={max_val:.6g}")
    # highlight the most severe single-case based on max_anomaly_value
    nonzero_max = trace_jdbc_anom_summary[trace_jdbc_anom_summary['max_anomaly_value'] > 0]
    if not nonzero_max.empty:
        worst = nonzero_max.sort_values('max_anomaly_value', ascending=False).iloc[0]
        lines.append("")
        lines.append(f"- Highest observed JDBC trace latency: {worst.cmdb_id} / {worst.trace_name} max={worst.max_anomaly_value:.6g} (earliest anomaly {worst.earliest_anomaly_timestamp}).")
    # quick guidance
    lines.append("")
    lines.append("Quick recommendations:")
    lines.append("- Ignore/verify the many error_rate entries with zero values before treating them as incidents (they were flagged because group_P95==0).")
    lines.append("- Prioritize investigating duration_p95/duration_mean groups with non-zero max values (e.g., docker_003, docker_001, docker_004, docker_002) for possible JDBC latency issues.")
    lines.append("- Correlate these latency spikes with DB metrics and application-side traces to find root cause.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                     trace_name  group_P95  anomaly_count earliest_anomaly_timestamp  first_anomaly_value  max_anomaly_value
0   docker_001  trace.self.jdbc.duration_mean   0.003171              2  2020-04-11 00:00:00+08:00             0.003879           0.003879
1   docker_007     trace.self.jdbc.error_rate   0.000000             30  2020-04-11 00:00:00+08:00             0.000000           0.000000
2   docker_001     trace.self.jdbc.error_rate   0.000000             30  2020-04-11 00:00:00+08:00             0.000000           0.000000
3   docker_006     trace.self.jdbc.error_rate   0.000000             30  2020-04-11 00:00:00+08:00             0.000000           0.000000
4   docker_002     trace.self.jdbc.error_rate   0.000000             30  2020-04-11 00:00:00+08:00             0.000000           0.000000
5   docker_008   trace.self.jdbc.duration_p95   0.003000             30  2020-04-11 00:00:00+08:00             0.003000           0.003000
6   docker_005     trace.self.jdbc.error_rate   0.000000             30  2020-04-11 00:00:00+08:00             0.000000           0.000000
7   docker_003     trace.self.jdbc.error_rate   0.000000             30  2020-04-11 00:00:00+08:00             0.000000           0.000000
8   docker_008     trace.self.jdbc.error_rate   0.000000             30  2020-04-11 00:00:00+08:00             0.000000           0.000000
9   docker_004     trace.self.jdbc.error_rate   0.000000             30  2020-04-11 00:00:00+08:00             0.000000           0.000000
10  docker_005  trace.self.jdbc.duration_mean   0.002755              2  2020-04-11 00:01:00+08:00             0.002838           0.002838
11  docker_008  trace.self.jdbc.duration_mean   0.002663              2  2020-04-11 00:01:00+08:00             0.002719           0.002719
12  docker_007  trace.self.jdbc.duration_mean   0.002679              2  2020-04-11 00:01:00+08:00             0.002682           0.002696
13  docker_006   trace.self.jdbc.duration_p95   0.004000              4  2020-04-11 00:02:00+08:00             0.004000           0.004000
14  docker_006  trace.self.jdbc.duration_mean   0.002657              2  2020-04-11 00:02:00+08:00             0.002701           0.002701
15  docker_007   trace.self.jdbc.duration_p95   0.003550              2  2020-04-11 00:02:00+08:00             0.004000           0.004000
16  docker_004   trace.self.jdbc.duration_p95   0.004000              3  2020-04-11 00:05:00+08:00             0.004000           0.005000
17  docker_003   trace.self.jdbc.duration_p95   0.100000              4  2020-04-11 00:06:00+08:00             0.100000           0.199000
18  docker_002   trace.self.jdbc.duration_p95   0.004138              2  2020-04-11 00:07:00+08:00             0.004250           0.004450
19  docker_001   trace.self.jdbc.duration_p95   0.004000              4  2020-04-11 00:07:00+08:00             0.006000           0.006000```
```